In [1]:
import sys
import os
import importlib
import anndata as ad
import pandas as pd
import scanpy as sc
import squidpy as sq
import logging
import numpy as np

/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/squidpy/gr/_utils.py:23: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  CAN_USE_SPARSE_ARRAY = Version(anndata.

In [2]:
sys.path.append(os.path.abspath("/Users/lakshmi_nccs/Desktop/NCCS/Projects/For_Publication/HNSCC_CosMx6k/helpers/"))
import helpers
importlib.reload(helpers)
import tcell_classifier
importlib.reload(tcell_classifier)

<module 'tcell_classifier' from '/Users/lakshmi_nccs/Desktop/NCCS/Projects/For_Publication/HNSCC_CosMx6k/helpers/tcell_classifier.py'>

In [3]:
data_dir = "/Users/lakshmi_nccs/Desktop/NCCS/Projects/Cosmx6k/results/final/P2/"
res_dir = "/Users/lakshmi_nccs/Desktop/NCCS/Projects/Cosmx6k/results/final/P3/"
if not (os.path.exists(res_dir)):
    os.makedirs(res_dir)

In [ ]:
#get distance to adata
ex = "all"
adata = ad.read_h5ad(data_dir + ex + "_adata_tcells_subtypes_cd8s.h5ad")
cd8s_distance = helpers.get_query_dist_target_all(adata, "tcell_type_resolved", "CD4- CD8+", "Tumor")
cd8s_touch = helpers.get_query_tn_target_all(adata, "tcell_type_resolved", "CD4- CD8+", "Tumor")
df_merged = cd8s_distance.obs.copy().merge(cd8s_touch.obs[['cell', 'TN_status']], on = 'cell', how = 'left')
cd8s_distance.obs = df_merged.copy()
cd8s_distance = helpers.stratify_spatial_bins(cd8s_distance, [0,250,500,800,6000], ["Q1: 0-250", "Q2: 250-500", "Q3: 500-800", "Q4: 800+"])
cd8s_distance.write_h5ad(res_dir + "cd8_tcells_dis_tn_tum.h5ad")

In [ ]:
ex = "all"
adata = ad.read_h5ad(data_dir + ex + "_adata_tcells_subtypes_cd8s.h5ad")
full_ans = [] 
for i in adata.obs['tma_sid'].unique():
    adata_sam = adata[adata.obs['tma_sid'] == i].copy()
    ans = helpers.get_spat_fingerprint_query(adata_sam, "tcell_type_resolved", "CD4- CD8+", 'ct', "TN")
    full_ans.append(ans)
full_ans = pd.concat(full_ans).fillna(0)
obs_keys = ['cell', 'tt', 'tma_sid', 'patient', 'tcelltype', 'query_obs', 'res_obs']
sp_adata = helpers.make_fingerprint_adata(full_ans,obs_keys)
sp_adata.write_h5ad(res_dir+"cd8_tcells_spfp_TN.h5ad")

In [ ]:
ex = "all"
adata = ad.read_h5ad(data_dir + ex + "_adata_tcells_subtypes_cd8s.h5ad")
full_ans = [] 
for i in adata.obs['tma_sid'].unique():
    adata_sam = adata[adata.obs['tma_sid'] == i].copy()
    ans = helpers.get_spat_fingerprint_query(adata_sam, "tcell_type_resolved", "CD4- CD8+", 'ct', 500)
    full_ans.append(ans)
full_ans = pd.concat(full_ans).fillna(0)
obs_keys = ['cell', 'tt', 'tma_sid', 'patient', 'tcelltype', 'query_obs', 'res_obs']
sp_adata = helpers.make_fingerprint_adata(full_ans,obs_keys)
sp_adata.write_h5ad(res_dir+"cd8_tcells_spfp_500.h5ad")

In [ ]:
sp_adata = ad.read_h5ad(res_dir+"cd8_tcells_spfp_TN.h5ad")
sp_adata_labelled = helpers.assign_dominant_contact_niche(sp_adata)
sp_adata_labelled.write_h5ad(res_dir+"cd8_tcells_spfp_TN_niche.h5ad")

In [ ]:
sp_adata = ad.read_h5ad(res_dir+"cd8_tcells_spfp_500.h5ad")
sp_adata_sf = helpers.cluster_sf(sp_adata,8, 8, 0.5)
adata_labelled = helpers.assign_niches_from_leiden(sp_adata_sf, 50,60,1)
sc.pl.umap(adata_labelled, color = ['spatial_niche'])
adata_labelled.write_h5ad(res_dir+"cd8_tcells_spfp_500_niche.h5ad")

In [ ]:
res = 1.5
ex = "cd8_tcells"
adata_cd8 = ad.read_h5ad(data_dir+ex+"_adata_res_" + str(res) + "_annot.h5ad")
cd8s_distance = ad.read_h5ad(res_dir + "cd8_tcells_dis_tn_tum.h5ad")
adata_spsf = ad.read_h5ad(res_dir+"cd8_tcells_spfp_500_niche.h5ad")
adata_spsf_TN = ad.read_h5ad(res_dir+"cd8_tcells_spfp_TN_niche.h5ad")
adata_spsf.obs.index.name = None
adata_spsf_TN.obs.index.name = None
df_merged = adata_cd8.obs.copy().merge(cd8s_distance.obs[['cell', 'Distance_to_Tumor', 'TN_status', "spatial_bin"]], on = 'cell', how = 'left')
df_merged = df_merged.merge(adata_spsf.obs[['cell', 'spatial_niche']], on = 'cell', how = 'left', suffixes=('', '_500'))
df_merged = df_merged.merge(adata_spsf_TN.obs[['cell', 'contact_niche']], on = 'cell', how = 'left', suffixes=('', '_TN'))
adata_cd8.obs = df_merged.copy()
adata_cd8.write_h5ad(res_dir+"cd8_spat_metrics.h5ad")

In [ ]:
cd8s_distance = ad.read_h5ad(res_dir+"cd8_spat_metrics.h5ad")
resp_df = pd.read_excel("/Users/lakshmi_nccs/Desktop/NCCS/Projects/Cosmx6k/processed_data/response.xlsx")
cd8s_distance.obs = cd8s_distance.obs.merge(resp_df, how='left', left_on='Patient', right_on='Patient')
cd8s_distance.obs['spatial_bin'].replace(['nan', None], 'Q4: 800+', inplace=True)
cd8s_distance.write_h5ad(res_dir+"cd8_dist_resp.h5ad")

In [ ]:
## Checking the microenvironment of tcells in plasma rich niche
sp_adata = ad.read_h5ad(res_dir+"cd8_tcells_spfp_500.h5ad")
#sp_adata = ad.read_h5ad(res_dir+"cd8_tcells_spfp_500_subtypes_micro.h5ad") # same as normal but with T_subtype
cd8s_distance = ad.read_h5ad(res_dir+"cd8_dist_resp.h5ad")
cd8s_distance_post = cd8s_distance[cd8s_distance.obs['TumorType'] == "Post-NIVO Primary"].copy()
cd8s_distance_post_resp = cd8s_distance_post[cd8s_distance_post.obs['Response'] == "Responder"].copy()
resp_plasma = cd8s_distance_post_resp[cd8s_distance_post_resp.obs['contact_niche'] == "Plasma Cells_rich_niche"].copy()

In [ ]:
sp_adata_subset = sp_adata[sp_adata.obs['cell'].isin(set(resp_plasma.obs['cell']))].copy()
micro_df = sp_adata_subset.to_df()
sp_adata_subset = sp_adata[~sp_adata.obs['cell'].isin(set(resp_plasma.obs['cell']))].copy()
micro_other = sp_adata_subset.to_df()

In [75]:
(micro_df.mean() /micro_other.mean()).sort_values(ascending=False)

Plasma Cells         11.959760
B Cells               1.688326
T Cells               1.510111
Endothelial Cells     1.161711
Myogenic Cells        1.124090
Mast Cells            1.097226
Macrophages           1.059553
Fibroblasts           0.983491
Neuronal Cells        0.751844
Dentritic Cells       0.577286
Neutrophils           0.206498
Tumor                 0.100740
Epithelial Cells      0.000000
dtype: float64

In [ ]:
ex = "all"
adata = ad.read_h5ad(data_dir + ex + "_adata_tcells_subtypes_cd4s.h5ad")
cd8s_distance = helpers.get_query_dist_target_all(adata, "tcell_type_resolved", "CD4+ CD8-", "Tumor")
cd8s_touch = helpers.get_query_tn_target_all(adata, "tcell_type_resolved", "CD4+ CD8-", "Tumor")
df_merged = cd8s_distance.obs.copy().merge(cd8s_touch.obs[['cell', 'TN_status']], on = 'cell', how = 'left')
cd8s_distance.obs = df_merged.copy()
cd8s_distance = helpers.stratify_spatial_bins(cd8s_distance, [0,250,500,800,6000], ["Q1: 0-250", "Q2: 250-500", "Q3: 500-800", "Q4: 800+"])
cd8s_distance.write_h5ad(res_dir + "cd4_tcells_dis_tn_tum.h5ad")